# China semi-balance

Builds the desk table that mixes:
1. SCI feedstock processing (national + province × ownership)
2. SCI local crude output (province → equity owner → oilfield)
3. Petro-Logistics FSU China pipeline routes (kbd)

Config: `config/china_semi_balance.yaml`. CLI: `python scripts/build_china_semi_balance.py`.

In [1]:
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from fetch_data import load_cached_frames, save_frames
from src.client import SciClient
from src.reports.china_semi_balance import (
    build_china_semi_balance,
    load_config,
    write_china_semi_balance,
)
from src.reports.fsu_pipeline import parse_fsu_china_pipeline, resolve_fsu_path
from src.transforms import parse_crude_output

cfg = load_config()
cfg

c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\.venv\Lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


{'months': {'start': '2026-01-01', 'end': '2026-06-01'},
 'barrels_per_mt': 7.33,
 'fsu': {'path': 'FSU-*_bpd_interim.xlsx',
  'sheet': 'Crude Oil',
  'routes': ['China via Kazakhstan', 'China via ESPO']},
 'output': {'path': 'data/reports/china_semi_balance.xlsx'}}

In [2]:
frames = load_cached_frames()
assert frames, "No cache — run: python fetch_data.py --refresh --history"

if "crude_output" not in frames or frames["crude_output"].empty:
    client = SciClient()
    client.authenticate()
    pkg = client.fetch_package()
    sheet = next(
        s for s in pkg["SheetList"]
        if "crude" in s["SheetName"].lower() and "output" in s["SheetName"].lower()
    )
    frames["crude_output"] = parse_crude_output(sheet)
    save_frames(frames)

print("processing_ownership", frames["processing_ownership"].shape)
print("crude_output", frames["crude_output"].shape)
frames["crude_output"].head()

processing_ownership (3252, 5)
crude_output (445, 6)


,month_start,province,metric_type,name,ownership,production_kt
0,2026-01-01,Anhui,province_total,Anhui,None,0.0
1,2026-01-01,Beijing,province_total,Beijing,None,0.0
2,2026-01-01,Chongqing,equity_owner,PetroChina,PetroChina,3.5
3,2026-01-01,Chongqing,field,Xinan Oilfield,PetroChina,3.5
4,2026-01-01,Chongqing,province_total,Chongqing,None,3.5


In [3]:
fsu_path = resolve_fsu_path(cfg["fsu"]["path"], search_dir=ROOT)
fsu = parse_fsu_china_pipeline(fsu_path, routes=tuple(cfg["fsu"]["routes"]))
print(fsu_path.name)
fsu[fsu["month_start"].between(cfg["months"]["start"], cfg["months"]["end"])]

FSU-2026-07_bpd_interim.xlsx


,month_start,route,bpd,kbd
84,2026-01-01,China via ESPO,604821.0,604.821
85,2026-02-01,China via ESPO,606846.0,606.846
86,2026-03-01,China via ESPO,550481.0,550.481
87,2026-04-01,China via ESPO,607892.0,607.892
88,2026-05-01,China via ESPO,607183.0,607.183
89,2026-06-01,China via ESPO,607184.0,607.184
175,2026-01-01,China via Kazakhstan,243999.0,243.999
176,2026-02-01,China via Kazakhstan,243902.0,243.902
177,2026-03-01,China via Kazakhstan,243763.0,243.763
178,2026-04-01,China via Kazakhstan,243245.0,243.245


In [4]:
rows, months = build_china_semi_balance(
    frames,
    fsu_path=cfg["fsu"]["path"],
    start=cfg["months"]["start"],
    end=cfg["months"]["end"],
    routes=cfg["fsu"]["routes"],
    search_dir=ROOT,
)
print(len(months), "months:", [m.strftime("%Y-%m") for m in months])
print(len(rows), "rows")
rows.head(20)

6 months: ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
167 rows


,section,sub_category,item,unit,value_kind,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01,2026-06-01
0,None,Processing volume,National feedstock primary processing volume (kt),kt,kt,63744.3,60036.0,61709.6,55673.2,54391.7,51037.5
1,None,Processing volume,SOE feedstock primary processing volume (kt),kt,kt,41255.0,38672.0,39438.7,35073.1,34532.3,33663.4
2,None,Processing volume,Independent refinery feedstock primary process...,kt,kt,22489.3,21364.0,22270.9,20600.1,19859.4,17374.1
3,Breakdown by province and equity owner,None,None,None,kt,NaN,NaN,NaN,NaN,NaN,NaN
4,None,FEEDSTOCK PRIMARY PROCESSING UNITS,Anhui,kt,kt,600.0,540.0,570.5,550.0,580.0,560.0
5,None,None,Sinopec,kt,kt,600.0,540.0,570.5,550.0,580.0,560.0
6,None,FEEDSTOCK PRIMARY PROCESSING UNITS,Beijing,kt,kt,680.0,640.0,659.1,570.0,570.0,572.0
7,None,None,Sinopec,kt,kt,680.0,640.0,659.1,570.0,570.0,572.0
8,None,FEEDSTOCK PRIMARY PROCESSING UNITS,Fujian,kt,kt,1350.0,2190.0,1955.7,1415.4,1340.0,1805.0
9,None,None,Sinopec,kt,kt,900.0,810.0,749.5,450.0,520.0,780.0


In [5]:
# Quick checks against the hand-built sample logic
nat = rows[rows["item"].astype(str).str.contains("National feedstock", na=False)].iloc[0]
print("National Jan kt", nat["2026-01-01"])

tianjin = rows[
    (rows["sub_category"] == "province") & (rows["item"] == "Tianjin")
].iloc[0]
print("Tianjin Jan kt", tianjin["2026-01-01"])

kz = rows[rows["item"] == "China via Kazakhstan"].iloc[0]
print("KZ Jan kbd", kz["2026-01-01"])

National Jan kt 63744.3
Tianjin Jan kt 3382.0
KZ Jan kbd 243.999


In [6]:
out = write_china_semi_balance(frames, config=cfg)
print("Wrote", out)

Wrote data\reports\china_semi_balance.xlsx
